# 10 - Random Forest

The second model, evaluated exactly like the logistic regression in notebook 09 so the two can be compared fairly. Still no test set: model comparison uses the 12 development patients and the same 4 grouped cross-validation folds; the 5 test patients stay sealed until every model is built and one is chosen.

## Why a random forest might behave differently from logistic regression on our features

A random forest averages many decision trees, each grown on a bootstrap sample of the beats and allowed to consider only a random subset of the features at each split. Compared with the linear model this changes several things that matter for *our* data.

**Reasons it might do better**

1. **Non-linear rules and interactions.** Logistic regression can only weigh features additively. The EDA showed multi-modal distributions (`rr_ratio` has a mode at 1.0 and another near 0.5; `dominant deflection` and `amp_min` have a separate inverted cluster), and the classes are defined by *combinations* ("early AND wide"). A tree can carve those regions directly. Where the linear model was blind - atrial and fusion beats that look normal in any single feature - interactions might help.
2. **No scaling, robust to outliers.** Trees only compare a feature with a threshold, so the 305-fold spread in feature scale and the heavy-tailed RR values are irrelevant: no scaler, no clipping.
3. **Collinearity is harmless.** The correlated amplitude features made the logistic weights unreadable; a forest just spreads the work between them.
4. **Lower variance from averaging** (bagging plus random feature subsets): one fully grown tree memorises its training beats, a forest of them far less so.

**Reasons it might do worse across patients**

5. **Trees cannot extrapolate.** Outside the range seen in training a tree's answer is constant, while a linear model keeps following its slope. A new patient whose amplitudes or rhythm lie outside the training patients' range is exactly where that bites.
6. **High capacity plus patient shift.** Fully grown trees can carve out regions that exist only for particular training patients, and the EDA showed relationships (amplitude vs label) that change sign between patients.
7. **Coarser, uncalibrated probabilities.** The output is the share of trees voting "abnormal", so the default 0.5 threshold does not mean the same thing as for logistic regression - which is why threshold-free metrics (ROC-AUC, PR-AUC) matter in this comparison.
8. **Randomness.** It uses a random seed and results can shift between runs, so the seed is fixed in advance (42) and its influence is measured below.

Points 1-4 are predictions to test (does it do better on the beats the linear model missed?); points 5-8 are risks to test (does it do worse on unusual patients, and how much does the seed matter?). The sections below check each instead of assuming.

## Ground rules for a fair comparison

- **Same data:** the 12 development patients, the same 4 grouped folds, the same 13 `MODEL_FEATURES`. The test patients are dropped from memory below.
- **Same evaluation code, metrics and tables** (`src/evaluation.py`), and the same 0.5 threshold for the threshold-based metrics.
- **Different preprocessing, because the models need different things:** the logistic regression gets clipped, standardised inputs; the forest gets the raw features.
- **Untuned defaults for both.** The forest: 300 fully grown trees, about 3 features tried per split, seed 42 chosen before any result was seen.
- **Uncertainty:** with 12 development patients, differences are judged with a patient-level bootstrap (resampling whole patients), not by eye.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from src.evaluation import (annotated_rhythm, flagged_rate_by_symbol, metrics_by_group, out_of_fold_probabilities,
                            patient_bootstrap, permutation_importance_by_fold, summarize)
from src.feature_extraction import MODEL_FEATURES, add_record_relative_features, load_beat_table
from src.models import make_logistic_regression, make_random_forest
from src.splitting import apply_split, load_split

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

table = add_record_relative_features(load_beat_table()).sort_values(["record", "r_peak_sample"]).reset_index(drop=True)
data = apply_split(table, load_split())
dev = data[data.split == "train"].copy()
del table, data                                   # the test rows are gone from this notebook
print(f"development set: {len(dev)} beats, {dev.record.nunique()} patients, {100 * dev.is_abnormal.mean():.1f}% abnormal")

## Training and prediction

The same `out_of_fold_probabilities` as for the logistic regression: train on three folds, predict the fourth, four times, so every development beat is scored by a model that never saw its patient. The logistic regression is recomputed here (it is deterministic) so both models sit in one notebook. Three rows: the logistic regression baseline, the forest, and the forest with `class_weight="balanced_subsample"` - the forest's counterpart of the one labelled variant tried for the linear model, not a tuning search.

In [ ]:
lr = out_of_fold_probabilities(make_logistic_regression, dev)
rf = out_of_fold_probabilities(make_random_forest, dev)                                   # seed 42, fixed in advance
rf_balanced = out_of_fold_probabilities(lambda: make_random_forest(class_weight="balanced_subsample"), dev)

rows = {
    "logistic regression (baseline)": summarize(dev.is_abnormal, lr),
    "random forest": summarize(dev.is_abnormal, rf),
    "random forest, balanced_subsample": summarize(dev.is_abnormal, rf_balanced),
}
comparison = pd.DataFrame(rows).T[["accuracy", "precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
print(f"No-skill PR-AUC (prevalence): {dev.is_abnormal.mean():.3f}   |   'always Normal' accuracy: {1 - dev.is_abnormal.mean():.3f}")
comparison

### Is the difference real? A patient-level bootstrap

Resample the 12 patients with replacement 1,000 times and score both models on each resample. Beats within a patient are not independent, so patients - not beats - are the unit that is resampled.

In [ ]:
boot = patient_bootstrap(dev, lr, rf, n_boot=1000, seed=0)
rows = []
for metric, label in [("roc_auc", "ROC-AUC"), ("pr_auc", "PR-AUC"), ("f1", "F1 at threshold 0.5")]:
    diff = boot[metric + "_b"] - boot[metric + "_a"]
    rows.append((label, diff.mean(), *np.percentile(diff, [2.5, 97.5]), (diff > 0).mean()))
pd.DataFrame(rows, columns=["metric (forest minus logistic regression)", "mean difference", "2.5th percentile",
                            "97.5th percentile", "share of resamples where the forest is better"]).round(3)

**Reading the pooled table and the bootstrap together** (default settings, seed 42):

- **The forest ranks better, but only ROC-AUC shows a clear gain.** ROC-AUC is 0.904 vs 0.762; in the patient-level bootstrap the gain is +0.127 (95% interval 0.002 to 0.286) with the forest ahead in 98% of resamples - the interval only just excludes zero. PR-AUC is 0.725 vs 0.680, but its interval (-0.102 to +0.138) is wide: **with 12 patients that difference cannot be told apart from noise.**
- **At the default threshold the forest is not better.** F1 is 0.612 vs 0.645, precision 0.64 vs 0.73 and the false-alarm rate 6.2% vs 4.0%, at about the same recall (0.59 vs 0.58). The bootstrap agrees: the forest's F1 is lower by about 0.03 and ahead in only 7.5% of resamples. Its probabilities are on a different scale, so 0.5 is simply a worse operating point for it.
- **The precision-recall curves cross** (left panel of the figure below). The logistic regression is more precise up to a recall of about 0.55; beyond about 0.65 the forest holds a precision near 0.47-0.49 out to a recall of 0.9, where the linear model has fallen to almost the no-skill line. So the forest can retrieve many of the abnormal beats the linear model cannot rank at all - at roughly a coin-flip's precision. Which model is "better" depends on which operating region you care about.
- **`balanced_subsample` reshapes the forest** (precision 0.73, false alarms 4.0%, F1 0.642, ROC-AUC 0.87): it lands almost exactly where the logistic regression's default sits. Class weighting and the threshold are decisions for later; this row only shows that the forest's operating point is not fixed by the model.

The honest summary: whether the forest wins depends on the metric, and none of the differences is large relative to the uncertainty.

### How much does the random seed matter?

The forest is random. Five more seeds, same data and folds (seed 42 stays the main model; it is not replaced by a better-looking one).

In [ ]:
seed_rows = {}
for seed in [0, 1, 2, 3, 4]:
    proba = out_of_fold_probabilities(lambda seed=seed: make_random_forest(random_state=seed), dev)
    seed_rows[f"seed {seed}"] = summarize(dev.is_abnormal, proba)
seed_table = pd.DataFrame(seed_rows).T[["precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float)
pd.concat([seed_table, seed_table.agg(["mean", "std", "min", "max"])]).round(3)

The forest's headline number moves with the seed: across seeds 0-4, F1 runs from 0.598 to 0.644 (mean 0.630, std 0.018) and precision from 0.61 to 0.72, while recall (0.584-0.588), ROC-AUC (0.894-0.908) and PR-AUC (0.713-0.730) barely change. The threshold-free metrics are stable; the threshold-based ones are not, because the errors are clustered in a few patients (records 202 and 232) whose beats flip across the 0.5 line together when a few trees change.

So a single-seed F1 must not be read to the second decimal - including seed 42's 0.612, which sits about one standard deviation below the seed mean - and **the seed-to-seed spread (about 0.02 in F1) is as large as the gap to the logistic regression.** The seed was fixed in advance and is kept; it is not being picked.

## Where do the two models differ?

Per cross-validation fold, per patient, and per beat type (LR = logistic regression, RF = random forest, both out-of-fold at threshold 0.5).

In [ ]:
def side_by_side(group_column, columns):
    lr_part, rf_part = metrics_by_group(dev, lr, group_column), metrics_by_group(dev, rf, group_column)
    return pd.concat({"LR": lr_part[list(columns)], "RF": rf_part[list(columns)]}, axis=1).astype(float).round(3)


print("Per cross-validation fold")
side_by_side("cv_fold", ["recall", "false_alarm_rate", "f1", "pr_auc"])

In [ ]:
print("Per patient")
info = metrics_by_group(dev, lr, "record")[["beats", "abnormal"]].astype(int)
pd.concat([pd.concat({"": info}, axis=1), side_by_side("record", ["recall", "false_alarm_rate", "pr_auc"])], axis=1)

In [ ]:
lr_symbols, rf_symbols = flagged_rate_by_symbol(dev, lr), flagged_rate_by_symbol(dev, rf)
by_symbol = pd.DataFrame({"label": lr_symbols.label, "beats": lr_symbols.beats,
                          "LR % flagged": lr_symbols.pct_flagged_abnormal, "RF % flagged": rf_symbols.pct_flagged_abnormal})
print("Share of each beat type flagged as abnormal: recall for the abnormal types, false-alarm rate for the normal ones")
by_symbol

In [ ]:
prevalence = dev.is_abnormal.mean()
fig, axes = plt.subplots(1, 3, figsize=(20, 5.8), gridspec_kw={"width_ratios": [1, 1.15, 1.15]})

ax = axes[0]
for name, proba, color in [("logistic regression", lr, "tab:blue"), ("random forest", rf, "tab:orange")]:
    precision, recall, _ = precision_recall_curve(dev.is_abnormal, proba)
    s = summarize(dev.is_abnormal, proba)
    ax.plot(recall, precision, color=color, label=f"{name} (PR-AUC {s['pr_auc']:.2f})")
    ax.scatter(s["recall"], s["precision"], color=color, edgecolor="black", zorder=3)
ax.axhline(prevalence, color="grey", linestyle="--", label=f"no skill ({prevalence:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall (dots = threshold 0.5)")
ax.legend(loc="upper right")

ax = axes[1]
patient_lr, patient_rf = metrics_by_group(dev, lr, "record"), metrics_by_group(dev, rf, "record")
order = list(patient_lr.index)
y = np.arange(len(order))
ax.barh(y - 0.2, patient_lr.loc[order, "pr_auc"].astype(float), height=0.4, color="tab:blue", label="logistic regression")
ax.barh(y + 0.2, patient_rf.loc[order, "pr_auc"].astype(float), height=0.4, color="tab:orange", label="random forest")
ax.set_yticks(y, [f"{r} ({100 * patient_lr.loc[r, 'abnormal'] / patient_lr.loc[r, 'beats']:.0f}% abn)" for r in order])
ax.invert_yaxis()
ax.set_xlabel("PR-AUC within the patient")
ax.set_title("Per patient: no uniform winner")
ax.legend(loc="lower right")

ax = axes[2]
order_symbols = ["V", "A", "F", "J", "a", "N", "R", "L"]
x = np.arange(len(order_symbols))
ax.bar(x - 0.2, by_symbol.loc[order_symbols, "LR % flagged"], width=0.4, color="tab:blue", label="logistic regression")
ax.bar(x + 0.2, by_symbol.loc[order_symbols, "RF % flagged"], width=0.4, color="tab:orange", label="random forest")
ax.set_xticks(x, [f"{s}\n{'abn' if by_symbol.loc[s, 'label'] == 'Abnormal' else 'norm'}" for s in order_symbols])
ax.set_ylabel("% flagged as abnormal")
ax.set_title("By beat type: recall (abnormal) / false alarms (normal)")
ax.legend()

fig.tight_layout()
fig.savefig("../results/figures/21_rf_vs_logreg.png", dpi=120)
plt.show()

Where the two models differ (all out-of-fold):

- **Per fold**, PR-AUC is higher for the forest in folds 0, 1 and 3 (0.74 vs 0.66, 0.61 vs 0.52, 0.95 vs 0.90) and lower in fold 2 (0.92 vs 0.98). F1 at 0.5 is lower for the forest in folds 0 and 1 and about equal in folds 2 and 3.
- **Per patient the picture is a split, not a sweep.** The forest is clearly better on **220** (PR-AUC 0.71 -> 0.96; recall on its atrial beats 4% -> 54%, still with no false alarms), **234** (0.12 -> 0.56) and **213** (0.60 -> 0.72, with more fusion beats caught), and slightly on 124. It is much worse on **232** (0.998 -> 0.65) and **202** (0.43 -> 0.06). The two models fail in *different* places; neither is uniformly better.
- **By beat type**, ventricular beats stay at 99% for both. Atrial beats are *not* better caught overall (8.6% vs 10.4% flagged); fusion beats improve (35% vs 20%); junctional stay at 1%. And the forest raises false alarms on Normal beats: `N` 7.1% vs 4.8%, and it begins to flag some right-bundle-branch beats (`R` 2.5% vs 0%).

So the forest's interactions do help in specific patients where a linear rule cannot separate the classes (record 220's atrial beats) - which supports prediction 1 - but the gain is patient-specific and comes with more false alarms elsewhere.

### Irregular rhythm again

The false alarms of the logistic regression clustered in atrial fibrillation and flutter. Same breakdown for both models (rhythm labels are used for analysis only, never as inputs).

In [ ]:
rhythm = annotated_rhythm(dev)
normal_beats = dev[dev.label == "Normal"].assign(rhythm=rhythm, lr_alarm=(lr >= 0.5).astype(int), rf_alarm=(rf >= 0.5).astype(int))
by_rhythm = normal_beats.groupby("rhythm").agg(normal_beats=("lr_alarm", "size"), lr_false_alarms=("lr_alarm", "sum"),
                                               rf_false_alarms=("rf_alarm", "sum"))
by_rhythm["LR false-alarm %"] = (100 * by_rhythm.lr_false_alarms / by_rhythm.normal_beats).round(1)
by_rhythm["RF false-alarm %"] = (100 * by_rhythm.rf_false_alarms / by_rhythm.normal_beats).round(1)
by_rhythm.sort_values("normal_beats", ascending=False)

The irregular-rhythm problem is not solved by the forest and is, if anything, worse: in atrial fibrillation it flags 25.2% of Normal beats (logistic regression 17.4%), in flutter 98.1% (93.3%), and even in sinus rhythm 3.1% (1.9%). It also creates a new one: 12.1% false alarms in sinus bradycardia (`(SBR`, 388 beats), which the linear model never flagged. Interactions on the same timing features cannot tell "one premature beat in a regular rhythm" from "an irregular rhythm" - that needs information neither model has, such as the variability of the surrounding intervals.

## What does the forest rely on?

**Permutation importance on unseen patients:** in each fold, fit on the other folds, then shuffle one feature at a time in the held-out fold and record how much PR-AUC drops. A large positive drop means the model needs that feature to score *new* patients; zero or negative means it does not (or is hurt by it). Development data only. For comparison, the forest's built-in impurity importance from one fit on all development beats.

In [ ]:
importance = permutation_importance_by_fold(make_random_forest, dev)
importance["mean"] = importance.mean(axis=1)
importance = importance.sort_values("mean", ascending=False)

impurity = pd.Series(make_random_forest().fit(dev[MODEL_FEATURES], dev.is_abnormal).feature_importances_, index=MODEL_FEATURES)
pd.DataFrame({"permutation, mean over folds": importance["mean"], "impurity (one fit, all dev)": impurity}).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6.2))
im = ax.imshow(importance.values, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="auto")
ax.set_xticks(range(importance.shape[1]), [f"fold {c}" if c != "mean" else "mean" for c in importance.columns])
ax.set_yticks(range(len(importance)), importance.index)
for i in range(importance.shape[0]):
    for j in range(importance.shape[1]):
        ax.text(j, i, f"{importance.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="drop in PR-AUC when the feature is shuffled")
ax.set_title("Random forest: permutation importance on held-out patients")
fig.tight_layout()
fig.savefig("../results/figures/22_rf_permutation_importance.png", dpi=120)
plt.show()

Permutation importance on unseen patients is very uneven across folds. `amp_max` matters enormously in fold 1 (0.31) and hardly at all elsewhere; `amp_min` matters mostly in fold 2 (0.30); `qrs_fwhm_ms` in folds 0 and 2; the relative amplitude features (`amp_std_rel`, `qrs_p2p_rel`) and `rr_ratio` contribute modestly and more evenly. Three features have zero or *negative* importance (`rr_post_rel`, `rr_post_s`, `amp_mean`): shuffling them does not hurt, or even helps, on unseen patients.

Two lessons. First, how much the forest relies on a feature depends heavily on which patients are held out - another view of patient shift. Second, the built-in impurity importance gives every feature a share of the credit (`rr_ratio` ranks third, `rr_pre_s` and `rr_post_s` get a visible share), whereas on held-out patients most timing features contribute little; permutation importance on unseen patients is the more honest guide.

## Testing my own explanations

The forest did much worse than the linear model on records 232 and 202 and much better on 220. Two candidate explanations were checked instead of asserted.

**1. Is it absolute amplitude (patient identity)?** Remove the five absolute-amplitude features and refit both models.

In [ ]:
absolute_amplitude = ["amp_mean", "amp_std", "amp_min", "amp_max", "qrs_p2p_mv"]
reduced = [f for f in MODEL_FEATURES if f not in absolute_amplitude]

ablation_rows, ablation_patient = {}, {}
for model_name, factory in {"LR": make_logistic_regression, "RF": make_random_forest}.items():
    for label, features in {"all 13 features": MODEL_FEATURES, "without absolute amplitude (8)": reduced}.items():
        proba = out_of_fold_probabilities(factory, dev, features=features)
        ablation_rows[f"{model_name}, {label}"] = summarize(dev.is_abnormal, proba)
        ablation_patient[f"{model_name}, {label}"] = metrics_by_group(dev, proba, "record")["pr_auc"].astype(float)

print("Pooled")
display_pooled = pd.DataFrame(ablation_rows).T[["precision", "recall", "f1", "false_alarm_rate", "roc_auc", "pr_auc"]].astype(float).round(3)
display_pooled

In [ ]:
print("PR-AUC per patient")
pd.DataFrame(ablation_patient).round(3)

**2. Is it extrapolation?** Trees cannot follow a trend beyond the training range. For each patient: what share of its beats lies outside the range of the training folds on at least one feature, and how does the forest score its abnormal and normal beats?

In [ ]:
lr_pr = metrics_by_group(dev, lr, "record")["pr_auc"].astype(float)
rf_pr = metrics_by_group(dev, rf, "record")["pr_auc"].astype(float)

rows = []
for record, part in dev.groupby("record"):
    fit = dev[dev.cv_fold != part.cv_fold.iloc[0]]
    low, high = fit[MODEL_FEATURES].min(), fit[MODEL_FEATURES].max()
    outside = ((part[MODEL_FEATURES] < low) | (part[MODEL_FEATURES] > high)).any(axis=1).mean()
    proba = rf.loc[part.index]
    rows.append({"record": record,
                 "% beats outside training range": round(100 * outside, 1),
                 "RF median P(abnormal): abnormal beats": proba[part.is_abnormal == 1].median(),
                 "RF median P(abnormal): normal beats": proba[part.is_abnormal == 0].median(),
                 "PR-AUC LR": lr_pr[record], "PR-AUC RF": rf_pr[record]})
pd.DataFrame(rows).set_index("record").sort_values("% beats outside training range", ascending=False).round(3)

**Two tests of my own explanations:**

- **Was it absolute amplitude (patient identity)? No.** I expected the forest's failures on 232 and 202 to come from leaning on absolute amplitude. Removing the five absolute-amplitude features makes *both* models worse (forest PR-AUC 0.725 -> 0.661, logistic regression 0.680 -> 0.611), and 232 and 202 stay bad without them (232: 0.65 -> 0.67; 202: 0.06 -> 0.07). So amplitude helps across the development patients and does not explain those two failures. (The sign-flip risk from notebook 08 concerns patients we cannot see in this cross-validation; this ablation cannot rule it out.)
- **Was it extrapolation (risk 5)? For 232, partly.** 51% of record 232's beats lie outside the training range on at least one feature (mostly `amp_std` and `qrs_p2p_mv`) - far more than any other patient bar 205 (30%) - and the forest gives its abnormal and normal beats almost the same probability (median 0.25 vs 0.30), while the linear model, which keeps following its slope, still ranks them (PR-AUC 0.998). But this cannot be the whole story: record 205 is also 30% out of range and the forest scores it well (0.94). For 202 extrapolation is not the cause at all (1% out of range); it is the fibrillation stretches from the previous section.

Both remain hypotheses that fit the evidence, not proven mechanisms.

## Summary

**Model:** random forest, 300 fully grown trees, scikit-learn defaults otherwise, seed 42, raw features; evaluated by the same grouped cross-validation on the 12 development patients as the logistic regression. The test set is untouched.

| pooled, out-of-fold | precision | recall | F1 | false alarms | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|
| logistic regression | 0.73 | 0.58 | 0.65 | 4.0% | 0.76 | 0.68 |
| random forest | 0.64 | 0.59 | 0.61 | 6.2% | 0.90 | 0.73 |

**Verdict: no clear winner.** The forest ranks abnormal above normal beats better (ROC-AUC gain borderline-significant in the patient bootstrap), but its PR-AUC gain is within noise, and at the default threshold it raises more false alarms and has a lower F1 - with seed-to-seed variation as large as the gap.

**What we learned**

1. Interactions help in specific patients where a linear rule cannot separate the classes (220's atrial beats, fusion beats in 213, ranking in 234), but atrial beats overall are still missed (8.6% flagged) and the gains come with extra false alarms.
2. Both models fail on irregular rhythm - the forest more so, and it adds a new failure in sinus bradycardia. This is a feature problem, not a model problem.
3. A patient outside the training range (232) defeats the forest but not the linear model; extrapolation is one plausible reason, not a proven one.
4. Absolute amplitude helps both models across development patients (the sign-flip caveat from notebook 08 still stands).
5. Everything must be reported per fold and per patient, with seed variation and patient-level uncertainty.

**Next:** gradient boosting is the last planned model. The bigger lever suggested by both models' errors is a rhythm-context feature (variability of the recent RR intervals) - a design decision to make deliberately, from development evidence.